In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ERROR_TYPES = [
    "none",
    "wrong_model",
    "wrong_brand",
    "wrong_variant",
    "fake_clone",
    "missing_model",
    "ambiguous_variant",
    "too_generic",
    "unidentifiable",
    "not_a_phone",
    "not_an_offer",
]


In [ ]:
cwd = Path.cwd()

if (cwd / "src" / "preprocessing" / "model_keys.py").exists():
    ROOT = cwd
elif (cwd.parent / "src" / "preprocessing" / "model_keys.py").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate src/preprocessing/model_keys.py. "
        "Run this notebook from the repository root or notebooks/."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"

from src.preprocessing.model_keys import predict_model_key

print("Project root:", ROOT)
print("Model-key rules imported successfully.")


In [ ]:
# Check the input files required for manual review

required_files = [
    DATA_DIR / "splits" / "validation.csv",
    DATA_DIR / "labels" / "gold_set_Aya.csv",
]

for path in required_files:
    print("FOUND:" if path.exists() else "MISSING:", path)


In [ ]:
# Inspect the input data structure

files_to_inspect = [
    DATA_DIR / "splits" / "validation.csv",
    DATA_DIR / "labels" / "gold_set_Aya.csv",
]

for path in files_to_inspect:
    if not path.exists():
        continue

    df = pd.read_csv(path)
    print(f"\n{path.name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Unique ad IDs:", df["ad_id"].nunique())


# Select validation data for manual review

In [ ]:
validation_df = pd.read_csv(DATA_DIR / "splits" / "validation.csv")
gold_df = pd.read_csv(DATA_DIR / "labels" / "gold_set_Aya.csv")

validation_ids = set(validation_df["ad_id"].dropna().astype(str))
gold_ids = set(gold_df["ad_id"].dropna().astype(str))

print("Validation and Gold Set overlap:", len(validation_ids & gold_ids))


In [ ]:
# Prepare validation ads and remove examples already in the Gold Set

review_candidates = validation_df.copy()
review_candidates["ad_id"] = review_candidates["ad_id"].astype(str)
gold_ids = set(gold_df["ad_id"].dropna().astype(str))

original_rows = len(review_candidates)
review_candidates = review_candidates[
    ~review_candidates["ad_id"].isin(gold_ids)
].drop_duplicates(subset="ad_id").copy()

print("Original validation rows:", original_rows)
print("Candidates after excluding Gold Set rows:", len(review_candidates))
print("Unique candidate ads:", review_candidates["ad_id"].nunique())

display(review_candidates[["ad_id", "title", "brand"]].head())


# Apply model rules to the validation ads

In [ ]:
# Apply the existing model extraction rules

review_candidates[["predicted_model_key", "match_rule"]] = (
    review_candidates.apply(
        lambda row: predict_model_key(
            row["title"],
            row["brand"],
            with_rule=True
        ),
        axis=1,
        result_type="expand"
    )
) 
display(review_candidates[[
    "ad_id",
    "title",
    "brand",
    "predicted_model_key",
    "match_rule"
]].head(10))

In [ ]:
# Identify ads with unknown predictions for manual review

pred = (
    review_candidates["predicted_model_key"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

rule = (
    review_candidates["match_rule"]
    .fillna("")
    .astype(str)
    .str.strip()
)

is_unknown_prediction = pred.isin(["", "unknown"])
is_unknown_brand = rule.eq("unknown_brand")

is_unknown_known_brand = (
    is_unknown_prediction & ~is_unknown_brand
)

review_candidates["review_bucket"] = "other_rule_review"

review_candidates.loc[
    is_unknown_known_brand,
    "review_bucket"
] = "no_match_known_brand"

review_candidates.loc[
    is_unknown_brand,
    "review_bucket"
] = "unknown_brand"


print("Review bucket counts:")
display(review_candidates["review_bucket"].value_counts())

print("\nUnknown predictions:", is_unknown_prediction.sum())
print("Unknown brand rules:", is_unknown_brand.sum())
print("Unknown predictions with known brand:", is_unknown_known_brand.sum())





In [ ]:
# Collect unknown and no-match cases for manual review

unknown_review = review_candidates[
    review_candidates["review_bucket"].isin([
        "no_match_known_brand",
        "unknown_brand",
    ])
].copy()

unknown_review["reviewed_model_key"] = ""
unknown_review["error_type"] = ""
unknown_review["notes"] = ""
review_df = unknown_review.copy()

print("Unknown and no-match cases:", len(review_df))
display(review_df[[
    "ad_id",
    "title",
    "brand",
    "predicted_model_key",
    "match_rule",
    "review_bucket",
]])


In [ ]:
print("Unknown and no-match ads:", len(review_df))
print("Cases by category:")
display(review_df["review_bucket"].value_counts())
display(review_df[[
    "ad_id",
    "title",
    "brand",
    "predicted_model_key",
    "match_rule",
    "review_bucket",
]].head(20))


# Review unknown-brand cases

In [ ]:
#reviewing unknown brands
unknown_brand_df = review_candidates[
    review_candidates["review_bucket"] == "unknown_brand"
].copy()

print("Number of unknown-brand cases:", len(unknown_brand_df))

display(
    unknown_brand_df[
        ["ad_id", "title", "brand", "predicted_model_key", "match_rule"]
    ].head(20)
)

In [ ]:
no_match_df = review_candidates[
    review_candidates["review_bucket"] == "no_match_known_brand"
].copy()

print("Number of no-match cases:", len(no_match_df))

display(
    no_match_df[
        [
            "ad_id",
            "title",
            "brand",
            "predicted_model_key",
            "match_rule",
        ]
    ]
)

# Find low-confidence predictions

In [ ]:
# Measure rule support using predictions on the Gold Set

gold_predictions = gold_df.apply(
    lambda row: predict_model_key(row["title"], row["brand"], with_rule=True),
    axis=1,
    result_type="expand",
)
gold_predictions.columns = ["predicted_model_key", "match_rule"]

gold_evaluation = pd.concat(
    [gold_df[["ad_id", "gold_model_key"]], gold_predictions],
    axis=1,
)
gold_evaluation = gold_evaluation[
    gold_evaluation["predicted_model_key"] != "unknown"
].copy()
gold_evaluation["is_correct"] = (
    gold_evaluation["predicted_model_key"] == gold_evaluation["gold_model_key"]
)

rule_stats = gold_evaluation.groupby("match_rule")["is_correct"].agg(
    support="size",
    correct="sum",
)
rule_stats["accuracy"] = rule_stats["correct"] / rule_stats["support"]

print("Rules used on Gold Set:", len(rule_stats))
print("Rules with fewer than 5 examples:", (rule_stats["support"] < 5).sum())
print("Gold Set accuracy on matched rows:", round(gold_evaluation["is_correct"].mean(), 3))
print("Rule support is used as a review signal; Gold Set accuracy is not an independent test score.")


In [ ]:
# Select known-brand ads with weakly supported or generic rules

known_df = review_candidates[
    ~review_candidates["review_bucket"].isin([
        "unknown_brand",
        "no_match_known_brand",
    ])
].copy()

known_df["rule_support_in_gold"] = (
    known_df["match_rule"].map(rule_stats["support"]).fillna(0).astype(int)
)
rules = known_df["match_rule"].fillna("").astype(str)
known_df["low_conf_reason"] = ""

for condition, reason in [
    (known_df["rule_support_in_gold"].eq(0), "never_seen_in_gold"),
    (rules.str.contains(r"fallback_pattern|numeric_pattern|ambiguous"), "weak_generic_rule"),
    (rules.str.contains(r"inferred_|arabic_|mate_30_pro_5g"), "hand_patched_rule"),
]:
    known_df.loc[condition, "low_conf_reason"] += reason + "+"

known_df["low_conf_reason"] = known_df["low_conf_reason"].str.rstrip("+")
low_pool = known_df[known_df["low_conf_reason"] != ""].copy()

print("Known-brand predictions:", len(known_df))
print("Low-confidence ads:", len(low_pool))
display(low_pool["low_conf_reason"].value_counts())


In [ ]:
# Prepare low-confidence ads for manual review

low_df = low_pool.copy()
low_df["review_bucket"] = "low_confidence"
low_df[["reviewed_model_key", "error_type", "notes"]] = ""

print("Low-confidence ads:", len(low_df))
print("Unique title groups:", low_df["title_group"].nunique())
display(low_df["brand"].value_counts(dropna=False))


In [ ]:
# Preview the low-confidence review cases

display(low_df[[
    "ad_id",
    "title",
    "brand",
    "predicted_model_key",
    "match_rule",
    "rule_support_in_gold",
    "low_conf_reason",
]])


# Combine review cases and save the CSV

In [ ]:
# Combine unknown, no-match, and low-confidence ads

full_review = pd.concat(
    [review_df, low_df],
    ignore_index=True,
    sort=False,
).drop_duplicates(subset="ad_id").copy()

print("Total unique ads for manual review:", len(full_review))
print("Cases by review category:")
display(full_review["review_bucket"].value_counts())


In [ ]:
# Keep the columns needed for manual review

review_columns = [
    "ad_id",
    "ad_url",
    "title",
    "brand",
    "price",
    "predicted_model_key",
    "match_rule",
    "confidence",
    "review_bucket",
    "rule_support_in_gold",
    "low_conf_reason",
    "reviewed_model_key",
    "error_type",
    "notes",
]

for column in review_columns:
    if column not in full_review.columns:
        full_review[column] = ""

for column in ["reviewed_model_key", "error_type", "notes"]:
    full_review[column] = full_review[column].fillna("")

# Convert relative Dubizzle paths into clickable listing links.
relative_url = full_review["ad_url"].fillna("").astype(str).str.startswith("/")
full_review.loc[relative_url, "ad_url"] = (
    "https://www.dubizzle.com.eg" + full_review.loc[relative_url, "ad_url"].astype(str)
)

full_review = full_review[review_columns].sort_values(
    ["review_bucket", "brand", "ad_id"],
    na_position="last",
).reset_index(drop=True)

display(full_review.head(10))


In [ ]:
# Save the CSV in the project data folder

OUTPUT_PATH = DATA_DIR / "labels" / "manual_review_candidates.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
full_review.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved review file:", OUTPUT_PATH)
print("Rows:", len(full_review))


### Manual review instructions

Open `data/labels/manual_review_candidates.csv` in Excel. Fill `reviewed_model_key`, `error_type`, and `notes`. Save a copy as `manual_review_candidates_reviewed.csv` in the same folder, then run the analysis cells below.

Use one `error_type` from the list defined in the first cell. Keep the final test set out of this manual review.


In [ ]:
REVIEWED_CSV = DATA_DIR / "labels" / "manual_review_candidates_reviewed.csv"

if not REVIEWED_CSV.exists():
    print("Manual review is not complete yet.")
    print("Save the completed Excel file as:", REVIEWED_CSV)
    reviewed = None
else:
    reviewed = pd.read_csv(REVIEWED_CSV, dtype={"ad_id": str}, encoding="utf-8-sig")
    for column in ["reviewed_model_key", "predicted_model_key", "error_type"]:
        reviewed[column] = reviewed[column].astype("string").str.strip().str.lower()

    missing_model = reviewed["reviewed_model_key"].isna() | reviewed["reviewed_model_key"].eq("")
    missing_type = reviewed["error_type"].isna() | reviewed["error_type"].eq("")
    invalid_type = ~reviewed["error_type"].isin(ERROR_TYPES) & ~missing_type

    print(f"Model labels completed: {(~missing_model).sum()} / {len(reviewed)}")
    print(f"Error types completed: {(~missing_type).sum()} / {len(reviewed)}")
    print(f"Invalid error types: {invalid_type.sum()}")

    if missing_model.any():
        print("Rows still needing a model label or an explicit unknown value:")
        display(reviewed.loc[missing_model, ["ad_id", "title", "error_type", "notes"]])


In [ ]:
if reviewed is None:
    print("Run this summary after adding the reviewed CSV.")
else:
    print("--- Review categories ---")
    display(reviewed["error_type"].value_counts(dropna=False).rename_axis("error_type").to_frame("rows"))

    labeled = reviewed[
        reviewed["reviewed_model_key"].notna()
        & reviewed["reviewed_model_key"].ne("")
    ].copy()
    labeled["model_key_matches"] = (
        labeled["predicted_model_key"] == labeled["reviewed_model_key"]
    )

    print(f"Rows with a reviewed model key: {len(labeled)} / {len(reviewed)}")
    if len(labeled):
        print(
            "Prediction agreement on labeled rows:",
            f"{labeled['model_key_matches'].mean():.2%}",
        )

    error_by_brand = pd.crosstab(
        reviewed["brand"].fillna("NO_BRAND"),
        reviewed["error_type"],
    )
    display(error_by_brand.assign(total=error_by_brand.sum(axis=1)).sort_values("total", ascending=False))


In [ ]:
if reviewed is None:
    print("Add the reviewed CSV before listing unresolved model cases.")
else:
    missing_model = reviewed["reviewed_model_key"].isna() | reviewed["reviewed_model_key"].eq("")
    if missing_model.any():
        print("Rows without a reviewed model key:")
        display(reviewed.loc[missing_model, [
            "ad_id", "title", "predicted_model_key", "error_type", "notes"
        ]])

    labeled = reviewed[~missing_model].copy()
    labeled["exact_match"] = labeled["predicted_model_key"] == labeled["reviewed_model_key"]
    print(f"Exact matches among labeled rows: {labeled['exact_match'].sum()} / {len(labeled)}")
    display(labeled.loc[~labeled["exact_match"], [
        "title", "predicted_model_key", "reviewed_model_key", "match_rule", "error_type"
    ]])
